1. Recombine neuronal and non-neuronal AnnData objects back into a single object
2. Update QC log for cells removed during clustering

In [17]:
from pathlib import Path
import tempfile
import os 

import scanpy as sc
import pandas as pd
import numpy as np
from dotenv import dotenv_values


In [2]:
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])
CLUSTERING_DIR = DATADIR / "5_clustering"
MODELDIR = CLUSTERING_DIR / 'all_control'
DATASETS_DIR = MODELDIR / "datasets"

ADATA_FP = DATASETS_DIR / 'raw_embed_propagated.h5ad'

LABELMAT_FP = CLUSTERING_DIR / "final_labelmat.csv"

CELL_HISTORY = DATADIR / "qc_log.csv" #QC log

### Load the labelmat

In [7]:
full_labelmat = pd.read_csv(LABELMAT_FP, index_col=0)
full_labelmat = full_labelmat.astype('category')
full_labelmat.shape

(920995, 5)

In [8]:
full_labelmat.head()

,L1,L2,L3,L4,L5
SC1-01_SAMN18754353_AAACCCACAACCAGAG,Gl1-1 Glut,Gl2-12,Gl3-112,Gl4-0403,Gl5-0403
SC1-01_SAMN18754353_AAACCCAGTCATCTAG,Gl1-1 Glut,Gl2-13,Gl3-115,Gl4-0418,Gl5-1043
SC1-01_SAMN18754353_AAACGAAAGCAAGCCA,Gl1-1 Glut,Gl2-12,Gl3-109,Gl4-0392,Gl5-0392
SC1-01_SAMN18754353_AAACGAAGTGCGGCTT,Gl1-1 Glut,Gl2-12,Gl3-112,Gl4-0405,Gl5-0405
SC1-01_SAMN18754353_AAACGAATCGCATAGT,Gl1-1 Glut,Gl2-12,Gl3-111,Gl4-0398,Gl5-1037


### Let's add it back to the full dataset

In [5]:
adata = sc.read_h5ad(ADATA_FP) 

but track the cells that were removed during clustering

In [9]:
removed_cells = list(set(adata.obs_names) - set(full_labelmat.index))
len(removed_cells)

1460

In [10]:

# Remove cell that were removed during clustering
adata = adata[full_labelmat.index].copy()


In [11]:
# write into obs
adata.obs = pd.concat([adata.obs, full_labelmat], axis=1)

In [12]:
# Check dtype and convert if it's an integer to avoid rounding errors during normalization
if np.issubdtype(adata.X.dtype, np.integer):
    adata.X = adata.X.astype(np.float32)
    
adata

AnnData object with n_obs × n_vars = 920995 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'cluster_id_label', 'supertype_id_label', 'subclass_id_label', 'class_id_label', 'C66_named', 'C185_named', 'C286_named', 'C465_named', 'ABC_class_propagated', 'ABC_class_confidence', 'to_be_removed', 'IsNeuronal', 'major_features', 'major_features_propagated', 'major_features_confidence', 'ABC_supertype_propagated', 'ABC_supertype_confidence', 'ABC_subclass_propagated', 'ABC_subclass_confidence', 'ABC_cluster_propagated', 'ABC_cluster_confidence', 'L1', 'L2', 'L3', 'L4', 'L5'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene', 'IsHvg'
    uns: 'neighbors'

In [13]:
#save
adata.write_h5ad(DATASETS_DIR / f'control_clustered.h5ad')

### Let's track the removed cells

In [18]:
# Load existing file
qc_log_df = pd.read_csv(CELL_HISTORY)

qc_log_df["IsClusteringRemoved"] = False
qc_log_df.loc[qc_log_df.CellID.isin(removed_cells), "IsClusteringRemoved"] = True

# Atomically replace cell history to avoid data corruption
output_path = CELL_HISTORY
output_dir = output_path.parent
# Create temp file in same directory
with tempfile.NamedTemporaryFile(mode="w", delete=False, dir=output_dir, suffix=".csv") as tmp:
    temp_path = Path(tmp.name)
    qc_log_df.to_csv(tmp.name, index=False)

os.replace(temp_path, output_path)